# Three ways to project a JSON-LD org graph into a table

The Cristin organization endpoint returns a typed JSON-LD graph. We want one flat table out of it:

> **`uri`, `en`, `nb`, `nn`** &mdash; one row per organization node, with its label in each language.

There are three ways to get there, with a clear trade-off between them:

| | speed | downside |
|---|---|---|
| **1. Structural traversal** | very fast | involved, and tied to the document's *structure* |
| **2. JSON-LD tooling** (`pyld`) | fast | tied to the JSON-LD *format* |
| **3. SPARQL** (`rdflib`) | slower | none, really &mdash; one simple query, independent of structure and format |

All three produce the same table; pick based on what you're optimising for.

In [1]:
# Setup: one fetch, shared by all three approaches.
#   pip install requests pandas rdflib "pyld==2.0.4"
#   (pin pyld==2.0.4 on Python 3.9; newer pyld needs 3.10+)
import json, requests, pandas as pd

URL = "https://api.nva.unit.no/cristin/organization/185.90.0.0"   # University of Oslo (big tree)
doc = requests.get(URL, headers={"Accept": "application/json"}, timeout=30).json()

COLS = ["uri", "en", "nb", "nn"]

## 1. Structural traversal &mdash; *very fast, but involved and structure-dependent*

In [2]:
# Walk the nested dicts yourself. No parsing, no libraries -> by far the fastest.
# The catch: you hand-code the graph's shape. It nests through BOTH hasPart (children,
# downward) and partOf (parents, upward), so a *correct* walk has to follow both. If the
# structure changes, this code changes with it.
def table_structural(doc):
    nodes = {}
    def visit(n):
        rec = nodes.setdefault(n["id"], {"uri": n["id"], "en": None, "nb": None, "nn": None})
        for lang, value in n.get("labels", {}).items():   # fill labels wherever a node appears
            if value:
                rec[lang] = value
        for neighbour in n.get("hasPart", []) + n.get("partOf", []):
            visit(neighbour)
    visit(doc)
    return pd.DataFrame(nodes.values()).reindex(columns=COLS)

df = table_structural(doc)
print(len(df), "rows")
df.head()

1072 rows


,uri,en,nb,nn
0,https://api.nva.unit.no/cristin/organization/1...,University of Oslo,Universitetet i Oslo,None
1,https://api.nva.unit.no/cristin/organization/1...,Faculty of Theology,Det teologiske fakultet,None
2,https://api.nva.unit.no/cristin/organization/1...,Det teologiske fakultetssekretariat,Det teologiske fakultetssekretariat,None
3,https://api.nva.unit.no/cristin/organization/1...,Fagseksjonen,Fagseksjonen,None
4,https://api.nva.unit.no/cristin/organization/1...,Gruppe A.T.,Gruppe A.T.,None


## 2. JSON-LD tooling &mdash; *fast, and structure-independent, but format-dependent*

In [10]:
# Let a JSON-LD library flatten the graph into a flat list of nodes. It finds every node
# no matter how the document nested, so you write no traversal. But it only works because
# the input is JSON-LD: it relies on the @context and the @language label container. Hand
# it Turtle, RDF/XML or a CSV and this approach doesn't apply.
from pyld import jsonld

LABEL = "https://nva.sikt.no/ontology/publication#label"   # what "labels" maps to in the @context

def table_pyld(doc):
    flat = jsonld.flatten(doc)                              # resolves the remote @context for you
    nodes = flat["@graph"] if isinstance(flat, dict) else flat
    rows = []
    for n in nodes:
        langs = {e["@language"]: e["@value"] for e in n.get(LABEL, [])}
        if langs:
            rows.append({"uri": n["@id"], **{l: langs.get(l) for l in ["en", "nb", "nn"]}})
    return pd.DataFrame(rows).reindex(columns=COLS)

df = table_pyld(doc)
print(len(df), "rows")
df.head()

1072 rows


,uri,en,nb,nn
0,https://api.nva.unit.no/cristin/organization/1...,Faculty of Theology,Det teologiske fakultet,None
1,https://api.nva.unit.no/cristin/organization/1...,Det teologiske fakultetssekretariat,Det teologiske fakultetssekretariat,None
2,https://api.nva.unit.no/cristin/organization/1...,Fagseksjonen,Fagseksjonen,None
3,https://api.nva.unit.no/cristin/organization/1...,Gruppe A.T.,Gruppe A.T.,None
4,https://api.nva.unit.no/cristin/organization/1...,Kompetansehevende studier,Kompetansehevende studier,None


## 3. SPARQL &mdash; *slower, but the simplest and independent of structure and format*

In [19]:
# Parse the document into RDF triples, then ask one query. Slowest, because it materialises
# the whole graph first -- but the query is trivial and robust: it doesn't care how the
# document nested, and the SAME query runs against any RDF you parsed (JSON-LD here, but
# Turtle / RDF-XML / N-Triples would work identically).
from rdflib import Graph

QUERY = """
PREFIX pub: <https://nva.sikt.no/ontology/publication#>
SELECT ?uri (SAMPLE(?enLabel) AS ?en) (SAMPLE(?nbLabel) AS ?nb) (SAMPLE(?nnLabel) AS ?nn)
WHERE {
  ?uri a pub:Organization .
  { ?uri pub:label ?enLabel . FILTER(lang(?enLabel) = "en") }
  UNION { ?uri pub:label ?nbLabel . FILTER(lang(?nbLabel) = "nb") }
  UNION { ?uri pub:label ?nnLabel . FILTER(lang(?nnLabel) = "nn") }
}
GROUP BY ?uri
"""

def table_sparql(doc):
    g = Graph()
    g.parse(data=json.dumps(doc), format="json-ld")        # also fetches the remote @context
    rows = [{c: (str(getattr(r, c)) if getattr(r, c) else None) for c in COLS}
            for r in g.query(QUERY)]
    return pd.DataFrame(rows).reindex(columns=COLS)

df = table_sparql(doc)
print(len(df), "rows")
df.head()

1072 rows


,uri,en,nb,nn
0,https://api.nva.unit.no/cristin/organization/1...,University of Oslo,Universitetet i Oslo,None
1,https://api.nva.unit.no/cristin/organization/1...,Faculty of Theology,Det teologiske fakultet,None
2,https://api.nva.unit.no/cristin/organization/1...,Det teologiske fakultetssekretariat,Det teologiske fakultetssekretariat,None
3,https://api.nva.unit.no/cristin/organization/1...,Fagseksjonen,Fagseksjonen,None
4,https://api.nva.unit.no/cristin/organization/1...,Gruppe A.T.,Gruppe A.T.,None


## The trade-off, measured

Same table, three costs. (The two RDF-based approaches also fetch the remote `@context` once
over the network; the warm-up call below absorbs that so we time pure compute.)

In [20]:
import timeit

for name, fn in [("1. structural", table_structural),
                 ("2. pyld     ", table_pyld),
                 ("3. sparql   ", table_sparql)]:
    fn(doc)                                                 # warm up caches / context fetch
    secs = timeit.timeit(lambda: fn(doc), number=5) / 5
    print(f"{name}  {secs*1000:8.1f} ms")

1. structural       3.7 ms
2. pyld          140.8 ms
3. sparql        323.8 ms
